# Ejercicio Práctico: Web Scraping de un E-Commerce de Libros (ETL con BeautifulSoup y Pandas)

En este ejercicio vas a construir por tu cuenta un flujo completo de **Extracción, Transformación y Carga (ETL)** utilizando `requests`, `BeautifulSoup` y `pandas` sobre la tienda online de práctica [Books to Scrape](https://books.toscrape.com/).

### Objetivo del ejercicio
Extraer información de múltiples categorías de libros para construir un `DataFrame` limpio con **4 variables clave por libro**:
1. **Título completo** (`str`)
2. **Precio en libras** (`float`, limpiando el símbolo `£`)
3. **Valoración en estrellas** (`str`, extraída de los atributos de clase)
4. **Disponibilidad en Stock** (`str`, limpiando saltos de línea y espacios)

---

## Paso 1: Importación de librerías y configuración

Importa las librerías necesarias (`time`, `requests`, `BeautifulSoup`, `os` y `pandas`) y define el diccionario `headers` con el `User-Agent`.

Setea el máximo de caracteres a 800 en pandas

In [1]:

import os
import time
import re
import requests
from bs4 import BeautifulSoup as bs
import pandas as pd


# Configuramos Pandas para que muestre textos largos sin cortarlos

pd.set_option('max_colwidth', 800)

# Definimos nuestro 'disfraz' de navegador (User-Agent) para evitar bloqueos 403
headers = {
"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36"
}

## Paso 2: Regla del Francotirador (Inspeccionar y probar con 1 solo libro)

Antes de crear ningún bucle, descarga la página principal `https://books.toscrape.com/`, abre el Inspector del navegador (`F12`) y localiza la estructura HTML de un libro:

In [2]:
import requests
import urllib3

# Evita advertencias molestas en la consola de VS Code
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

headers = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

url_prueba = "https://books.toscrape.com/"

# Añadimos verify=False para saltarnos el bloqueo del Mac en VS Code
response = requests.get(url_prueba, headers=headers, timeout=10, verify=False)
response.raise_for_status()

print("¡Petición exitosa desde VS Code! Código:", response.status_code)


# 2. Creamos la sopa
soup = bs(response.content, "html.parser")

# 3. Buscamos todas las tarjetas de libros y seleccionamos el primero (índice 0) para probar
pods = soup.find_all("article", class_="product_pod")
print(f"Total de libros encontrados en la página: {len(pods)}")

primer_libro = pods[0]


¡Petición exitosa desde VS Code! Código: 200
Total de libros encontrados en la página: 20


In [3]:
# Probamos la extracción y limpieza de los 4 datos sobre 'primer_libro'

# 1. TÍTULO

titulo = primer_libro.find("h3").find("a")['title']


# 2. PRECIO: Quitamos el símbolo '£' y convertimos a float

precio = float(soup.select_one('p.price_color').text.strip().replace('£', ''))

# 3. ESTRELLAS
estrellas_texto = soup.select_one('p.star-rating')['class'][1]  
conversor = {'One': 1, 'Two': 2, 'Three': 3, 'Four': 4, 'Five': 5}
estrellas = conversor.get(soup.select_one('p.star-rating')['class'][1], 0)

# 4. STOCK: Usamos strip=True en get_text() para limpiar los saltos de línea

stock_texto = soup.find('p', class_='instock availability')
stock = stock_texto.get_text(strip=True) if stock_texto else "No disponible"

print(f"Título:    {titulo}")
print(f"Precio:    {precio}")
print(f"Estrellas: {estrellas}")
print(f"Stock:     {stock}")

Título:    A Light in the Attic
Precio:    51.77
Estrellas: 3
Stock:     In stock


## Paso 3: Extracción Multi-Página con Bucle y `time.sleep`

Ahora que sabemos extraer los 4 datos de una tarjeta, vamos a recorrer **3 categorías distintas de la tienda** (*Mystery*, *Science Fiction* y *Travel*) y a extraer los primeros 10 libros de cada una.

* **Regla ética y técnica:** Recuerda añadir `time.sleep(1)` al final de cada página para no saturar el servidor.

In [12]:
# Lista de tuplas con (Nombre_Categoria, URL) a scrapear
urls_categorias = [
    ("Misterio",        "https://books.toscrape.com/catalogue/category/books/mystery_3/index.html"),
    ("Ciencia Ficción", "https://books.toscrape.com/catalogue/category/books/science-fiction_16/index.html"),
    ("Viajes",          "https://books.toscrape.com/catalogue/category/books/travel_2/index.html")
]

# Inicializamos el diccionario vacío donde acumularemos los datos de todas las páginas
datos_libros = {
    "Categoria": [],
    "Titulo": [],
    "Precio_Libras": [],
    "Estrellas": [],
    "Stock": []
}

# BUCLE EXTERIOR: recorre cada categoría
for categoria, url in urls_categorias:
# Bloque fijo de petición y BeautifulSoup
 
    response = requests.get(url)
    response.encoding = 'utf-8'
    if response.status_code !=200:
        print (f"Error al acceder a {url}")
        continue

    soup = bs(response.text, "html.parser") #¿¿¿Por qué .text???

    time.sleep(1)

    # BUCLE INTERIOR: recorre los primeros 10 libros de esa categoría
    books = soup.find_all("article", class_=("product_pod"))

    for book in books[:10]:

        # 1. Título
        
        titulo = book.find("h3").find("a")['title']

        # 2. Precio
        precio_texto = book.find("p", class_="price_color").text.strip()
        precio = float(precio_texto.replace('£', ''))

        # 3. Estrellas

        clases_estrellas = book.find("p", class_="star-rating")['class']
        estrellas = clases_estrellas[1] 

        #4 Stock

        stock_texto = book.find("p", class_="instock availability").text.strip()

        # Guardamos los datos en las listas de nuestro diccionario
        datos_libros["Categoria"].append(categoria)
        datos_libros["Titulo"].append(titulo)
        datos_libros["Precio_Libras"].append(precio)
        datos_libros["Estrellas"].append(estrellas)
        datos_libros["Stock"].append(stock_texto)

    # Esperamos 1 segundo antes de pedir la siguiente página

    time.sleep(1.2)
    print("¡Extracción completada con éxito!")

¡Extracción completada con éxito!
¡Extracción completada con éxito!
¡Extracción completada con éxito!


In [13]:
print(datos_libros)

{'Categoria': ['Misterio', 'Misterio', 'Misterio', 'Misterio', 'Misterio', 'Misterio', 'Misterio', 'Misterio', 'Misterio', 'Misterio', 'Ciencia Ficción', 'Ciencia Ficción', 'Ciencia Ficción', 'Ciencia Ficción', 'Ciencia Ficción', 'Ciencia Ficción', 'Ciencia Ficción', 'Ciencia Ficción', 'Ciencia Ficción', 'Ciencia Ficción', 'Viajes', 'Viajes', 'Viajes', 'Viajes', 'Viajes', 'Viajes', 'Viajes', 'Viajes', 'Viajes', 'Viajes'], 'Titulo': ['Sharp Objects', 'In a Dark, Dark Wood', 'The Past Never Ends', 'A Murder in Time', 'The Murder of Roger Ackroyd (Hercule Poirot #4)', 'The Last Mile (Amos Decker #2)', 'That Darkness (Gardiner and Renner #1)', 'Tastes Like Fear (DI Marnie Rome #3)', 'A Time of Torment (Charlie Parker #14)', 'A Study in Scarlet (Sherlock Holmes #1)', 'Mesaerion: The Best Science Fiction Stories 1800-1849', 'Join', "William Shakespeare's Star Wars: Verily, A New Hope (William Shakespeare's Star Wars #4)", 'The Project', 'Soft Apocalypse', 'Sleeping Giants (Themis Files #1)',

## Paso 4: Creación del DataFrame, Exploración y Guardado en CSV

1. Convierte el diccionario `datos_libros` en un `DataFrame` de Pandas llamado `df_libros`.
2. Comprueba sus dimensiones (`.shape`) y tipos de datos (`.dtypes`).
3. Calcula el **precio medio por categoría** usando `.groupby()`.
4. Guarda el resultado en `data/libros_scrapeados.csv` con separador `;` y sin índice (`index=False`).

In [16]:
# 1. Creamos el DataFrame a partir del diccionario recolectado

df = pd.DataFrame(datos_libros)
# 2. Comprobamos dimensiones y tipos de datos
df.shape
df.info()
# Mostramos los 10 primeros elementos del DataFrame
df
#Voy a poner Categoria como index

df.set_index("Categoria")

<class 'pandas.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Categoria      30 non-null     str    
 1   Titulo         30 non-null     str    
 2   Precio_Libras  30 non-null     float64
 3   Estrellas      30 non-null     str    
 4   Stock          30 non-null     str    
dtypes: float64(1), str(4)
memory usage: 1.3 KB


,Titulo,Precio_Libras,Estrellas,Stock
Categoria,,,,
Misterio,Sharp Objects,47.82,Four,In stock
Misterio,"In a Dark, Dark Wood",19.63,One,In stock
Misterio,The Past Never Ends,56.50,Four,In stock
Misterio,A Murder in Time,16.64,One,In stock
Misterio,The Murder of Roger Ackroyd (Hercule Poirot #4),44.10,Four,In stock
Misterio,The Last Mile (Amos Decker #2),54.21,Two,In stock
Misterio,That Darkness (Gardiner and Renner #1),13.92,One,In stock
Misterio,Tastes Like Fear (DI Marnie Rome #3),10.69,One,In stock
Misterio,A Time of Torment (Charlie Parker #14),48.35,Five,In stock


In [21]:
# 3. Precio medio, mínimo y máximo por categoría usando .groupby()
resumen_precios = df.groupby('Categoria')['Precio_Libras'].agg(['mean', 'min', 'max'])

resumen_precios = resumen_precios.rename(columns={
    'mean': 'Precio medio',
    'min': 'Precio mínimo',
    'max': 'Precio máximo'
})

print(resumen_precios)

                 Precio medio  Precio mínimo  Precio máximo
Categoria                                                  
Ciencia Ficción        28.584          10.65          48.74
Misterio               32.859          10.69          56.50
Viajes                 41.166          23.21          56.88


In [28]:
# 4. Guardamos el DataFrame en un archivo CSV limpio usando la librería 'os'
import os 

ruta_completa = "/Users/jorgeshz/Data Science Bootcamp/DS-Repo-JorgeSanchez/03_Data_Analysis/01_Sources/02_Web_Scraping/01_Teoria/01_Soup/data/libros.csv"

# Guardamos el archivo sin el índice para evitar columnas vacías innecesarias
df.to_csv(ruta_completa, index=False, encoding="utf-8-sig")